In [21]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline

# Problem Overview
## Classification of Hypertrophic Cardiomyopathy (Cardiomegaly)

## Task Description

Task was to prepare a solution for the **classification problem** of detecting **hypertrophic cardiomyopathy (cardiomegaly)** based on the provided features.  
I used one of the **classical machine learning methods**.

The goal was to build a model capable of correctly distinguishing between a **healthy heart** and a **diseased heart** using the available data.

The repository contains a dataset stored in a **CSV** file, which includes selected geometric and imaging features used as the basis for further analysis.  
I prepared code that:

1. Loads the data from the `task_data.csv` file.  
2. Splits the data into training and test sets.  
3. Performs preprocessing (e.g., standardization, normalization, etc.).  
4. Trains one or more selected models.  
5. Evaluates the solution using [cross-validation]. 
6. Evaluates the solution on the test dataset (e.g., using accuracy, precision, recall, F1-score).  
7. Provides a brief description of the chosen approach.

---

## Data (CSV)

The first column contains a **photo ID**, and the second column indicates whether the heart was diagnosed with cardiomegaly:  
- `1` – positive diagnosis (diseased heart)  
- `0` – negative diagnosis (healthy heart)  



# Data Loading 

In [13]:
# Import the data from the CSV file
# The file "task_data.csv" contains the complete dataset that we’ll split
# into training and testing parts - 80% for training and 20% for testing.
data = pd.read_csv("task_data.csv")
data.head()

# Zamiana przecinków na kropki w kolumnach tekstowych
for col in data.columns:
    if data[col].dtype == object:
        data[col] = data[col].str.replace(',', '.', regex=False)

# Usuń spacje z początku i końca nazw kolumn
data.columns = data.columns.str.strip()

# Data spliting into training and test sets

In [17]:
# Select the columns (features) that the model will use to learn
# These include numeric values describing each heart
X = data[[
    "Heart width", "Lung width", "CTR - Cardiothoracic Ratio", "xx", "yy", "xy", "normalized_diff", "Inscribed circle radius", "Polygon Area Ratio", "Heart perimeter", "Heart area", "Lung area"
]]

# Select the target column ("Cardiomegaly"), which represents the value the model aims to predict
# Based on the input features in X, the model will learn to determine whether a heart diseased (1) or not (0)
y = data["Cardiomegaly"]

# Split the dataset into training (80%) and testing (20%) parts
# This allows us to train the model on one portion and test it on unseen data.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing performing

In [18]:
# Create a StandardScaler object to normalize numeric data
# Scaling ensures that features are on a similar scale,
# which helps many algorithms train faster and perform better.
scaler = StandardScaler()

# Fit the scaler on the training data and apply it to both training and test sets
X_scaled_train = scaler.fit_transform(X_train)
X_scaled_test = scaler.transform(X_test)

# Model Training

- Performed preprocessing.
- Trained selected models.
- Evaluated the solution using cross-validation.

## K-Nearest Neighbors (KNN) Classifier

In [23]:
# Define a Pipeline that bundles preprocessing (scaling) and the model
# The scaler is fit ONLY on the training portion inside each CV fold,
# then applied to the validation fold — this prevents data leakage.
# We choose "random" hiperparameters at our own discretion
pipe_knn = Pipeline(steps=[
    ("scaler", StandardScaler()),     # Standardize features inside each CV fold
    ("model", KNeighborsClassifier(   # K-Nearest Neighbors classifier
        n_neighbors = 3,                # Number of neighbors used for prediction
        weights='distance',           # Weight controls how much influence each neighbor has when making a prediction.
        metric='manhattan'            # Look up different distance formulas in documentation
    ))
])

# Fit the Pipeline on the training data
# Internally, the scaler will be fit to X_train and the model will learn from the scaled data.
pipe_knn.fit(X_train, y_train)

# Evaluate the model using cross-validation WITHOUT leakage
# For each fold, the scaler is fit on the fold’s training split and applied to its validation split.
cv_score = np.round(cross_val_score(pipe_knn, X_train, y_train), 2)

# Display detailed results
# Shows accuracy for each fold, the mean accuracy, and the standard deviation.
# Lower standard deviation indicates more consistent performance across folds.
print("Scores of training data cross-validation (each fold):")
list(map(print, cv_score))
print(f"\nCross-validation mean score: {np.mean(cv_score):.3}")
print(f"Standard deviation of CV score: {np.std(cv_score):.3f}")

Scores of training data cross-validation (each fold):
0.83
0.83
0.67
0.83
1.0

Cross-validation mean score: 0.832
Standard deviation of CV score: 0.104


In [24]:
# Define a grid of hyperparameters to test
# Each combination of these parameters will be evaluated to find the best-performing model.
param_grid = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15],  # Number of neighbors to consider
    "model__weights": ["uniform", "distance"],   # How neighbors contribute to the prediction
    "model__metric": ["minkowski", "manhattan", "euclidean", "chebyshev"],  # Distance metrics to test
}

# Set up the cross-validation strategy
# RepeatedStratifiedKFold splits the data into several folds while preserving class balance.
# The process is repeated multiple times (100 here) to get more stable results.
rskf = RepeatedStratifiedKFold(
    n_splits=5,      # Number of folds per repetition
    n_repeats=100,   # Number of times to repeat the process
    random_state=None  # Random seed (None = random each run)
)

# Define a Pipeline combining the scaler and the model
# This ensures that scaling happens *within* each CV fold, preventing data leakage.
pipe_knn = Pipeline(steps=[
    ("scaler", StandardScaler()),           # Standardize data inside each CV fold
    ("model", KNeighborsClassifier())       # The KNN model to be optimized
])

# Initialize the Grid Search for the KNN model
# GridSearchCV will train and evaluate a model for every combination of parameters in param_grid,
# using the defined cross-validation strategy.
grid_search = GridSearchCV(
    estimator=pipe_knn,          # The pipeline (scaler + model)
    param_grid=param_grid,       # The grid of parameters to test
    scoring="accuracy",          # Metric used to evaluate model performance
    cv=rskf,                     # Cross-validation strategy
    verbose=1,                   # Display progress in the console
    n_jobs=-1                    # Use all available CPU cores for faster processing
)

# Train (fit) the grid search on the raw training data
# The scaler will be fit automatically within each fold.
grid_search.fit(X_train, y_train)

# Display the best results
# best_params_ shows which combination performed the best,
# best_score_ shows the corresponding average accuracy.
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best accuracy (averaged CV): {grid_search.best_score_:.4f}")

Fitting 500 folds for each of 48 candidates, totalling 24000 fits
Best parameters: {'model__metric': 'manhattan', 'model__n_neighbors': 5, 'model__weights': 'uniform'}
Best accuracy (averaged CV): 0.8303


In [25]:
# Refine the hyperparameter grid based on previous search results
# We now focus on narrower ranges around the most promising values to fine-tune the model.

param_grid = {
    "model__n_neighbors": [4, 5, 6],  
    "model__weights": ["uniform"],  
    "model__metric": ["manhattan"], 
}

rskf = RepeatedStratifiedKFold(
    n_splits=5,      
    n_repeats=100,   
    random_state=None  
)

pipe_knn = Pipeline(steps=[
    ("scaler", StandardScaler()),           
    ("model", KNeighborsClassifier())     
])

grid_search = GridSearchCV(
    estimator=pipe_knn,         
    param_grid=param_grid,     
    scoring="accuracy",         
    cv=rskf,                 
    verbose=1,                 
    n_jobs=-1                    
)

grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best accuracy (averaged CV): {grid_search.best_score_:.4f}")

Fitting 500 folds for each of 3 candidates, totalling 1500 fits
Best parameters: {'model__metric': 'manhattan', 'model__n_neighbors': 6, 'model__weights': 'uniform'}
Best accuracy (averaged CV): 0.8463


In [27]:
# We now enter optimized parameters into out model and check for improved accuracy score

pipe_knn = Pipeline(steps=[
    ("scaler", StandardScaler()),    
    ("model", KNeighborsClassifier(  
        n_neighbors = 6,              
        weights='uniform',         
        metric='manhattan'        
    ))
])

pipe_knn.fit(X_train, y_train)

cv_score = np.round(cross_val_score(pipe_knn, X_train, y_train), 2)

print("Scores of training data cross-validation (each fold):")
list(map(print, cv_score))
print(f"\nCross-validation mean score: {np.mean(cv_score):.3}")
print(f"Standard deviation of CV score: {np.std(cv_score):.3f}")

Scores of training data cross-validation (each fold):
1.0
1.0
0.83
0.67
1.0

Cross-validation mean score: 0.9
Standard deviation of CV score: 0.133


## Decision Tree

In [28]:
# We now move on to the Decision Tree.
# Since Decision Trees are not sensitive to feature scaling, we’ll not include data normalization and not use pipeline 

clf_tree = DecisionTreeClassifier(
    max_depth=7,            # Limits how deep the tree can grow (to prevent overfitting)
    criterion='log_loss',    # Measures the quality of a split using information gain
    min_samples_split=7,    # Minimum number of samples required to split an internal node
    min_samples_leaf=5,     # Minimum number of samples required to be at a leaf node
    class_weight=None       # No weighting — all classes are treated equally
)

clf_tree.fit(X_train, y_train)

cv_score = np.round(cross_val_score(clf_tree, X_train, y_train), 2)

print(f"Scores of training data cross-validation (each fold):")
list(map(print, cv_score))
print(f"\nCross-validation mean score: {np.mean(cv_score):.3}")
print(f"Standard deviation of CV score: {np.std(cv_score):.3f}")


Scores of training data cross-validation (each fold):
1.0
0.83
0.67
0.83
0.6

Cross-validation mean score: 0.786
Standard deviation of CV score: 0.140


## Support Vector Machine (SVM)

In [29]:
# We now move on to the Support Vector Classifier (SVC).
# Since SVMs are sensitive to feature scaling, we’ll include data normalization 
# directly in a Pipeline to ensure proper preprocessing and prevent any data leakage during cross-validation.

pipe_svc = Pipeline(steps=[
    ("scaler", StandardScaler()),   
    ("model", SVC(                  # Support Vector Classifier
        kernel="rbf",               # RBF kernel captures non-linear decision boundaries
        C=3,                        # Regularization strength (higher = tighter fit to training data)
        gamma="scale",              # Kernel width; 'scale' adapts to data variance
        class_weight=None           # Treat classes equally (no re-weighting)
    ))
])

pipe_svc.fit(X_train, y_train)

cv_score = np.round(cross_val_score(pipe_svc, X_train, y_train), 2)

print("Scores of training data cross-validation (each fold):")
list(map(print, cv_score))
print(f"\nCross-validation mean score: {cv_score.mean():.3f}")
print(f"Standard deviation of CV score: {cv_score.std():.3f}")

Scores of training data cross-validation (each fold):
1.0
0.83
0.67
0.83
1.0

Cross-validation mean score: 0.866
Standard deviation of CV score: 0.124


## Logistic Regression

In [30]:
# Same with logistic regression 

pipe_log = Pipeline(steps=[
    ("scaler", StandardScaler()),           # Normalize numeric features per fold
    ("model", LogisticRegression(           # Logistic Regression for binary classification
        C=1,                              # Inverse regularization strength (higher = weaker regularization)
        penalty="l1",                       # L1 regularization (drives some coefficients to exactly zero)
        solver="liblinear",                 # Solver compatible with L1 penalty
        max_iter=1000,                      # Ensure convergence
        class_weight=None                   # Treat classes equally
    ))
])

pipe_log.fit(X_train, y_train)

cv_score = np.round(cross_val_score(pipe_log, X_train, y_train), 2)

print("Scores of training data cross-validation (each fold):")
list(map(print, cv_score))
print(f"\nCross-validation mean score: {cv_score.mean():.3f}")
print(f"Standard deviation of CV score: {cv_score.std():.3f}")

Scores of training data cross-validation (each fold):
0.83
0.67
0.67
0.83
0.8

Cross-validation mean score: 0.760
Standard deviation of CV score: 0.074


## Random Forest Classifier

In [31]:
# And Random Forest

clf_rf = RandomForestClassifier(
    max_depth=6,                     # Limit the depth of each tree to prevent overfitting
    min_samples_split=6,             # Minimum number of samples required to split a node
    n_estimators=125,                # Number of trees in the forest
    min_samples_leaf=2,              # Minimum number of samples required to be at a leaf node
    max_features='sqrt'              # Number of features to consider when looking for the best split
)

clf_rf.fit(X_train, y_train)

cv_score = np.round(cross_val_score(clf_rf, X_train, y_train), 2)

print(f"Scores of training data cross-validation (each fold):")
list(map(print, cv_score))
print(f"\nCross-validation mean score: {np.mean(cv_score):.3f}")
print(f"Standard deviation of CV score: {np.std(cv_score):.3f}")


Scores of training data cross-validation (each fold):
1.0
0.83
0.83
0.83
1.0

Cross-validation mean score: 0.898
Standard deviation of CV score: 0.083


### Models score

| Classifier           | Accuracy |
|----------------------|----------|
| KNN                  | 90.0%    |
| Decision Tree        | 78.6%    |
| SVM                  | 86.6%    |
| Logistic Regression  | 76.0%    |
| Random Forest        | 89.8%    |

We achieved the highest accuracy with the KNN model, so now we’ll apply it to the test dataset to see how well it performs on unseen data.

## Evaluating the model on the test dataset

In [33]:
# Predictions: use raw X_test
# (Pipelines include preprocessing, so no manual scaling of X_test)
y_pred_knn  = pipe_knn.predict(X_test)   
y_pred_svc  = pipe_svc.predict(X_test)   
y_pred_log  = pipe_log.predict(X_test)   
y_pred_tree = clf_tree.predict(X_test)
y_pred_rf   = clf_rf.predict(X_test)

# Model evaluation: calculate accuracy for each model separately
acc_knn  = accuracy_score(y_test, y_pred_knn)
acc_tree = accuracy_score(y_test, y_pred_tree)
acc_svc  = accuracy_score(y_test, y_pred_svc)
acc_log  = accuracy_score(y_test, y_pred_log)
acc_rf   = accuracy_score(y_test, y_pred_rf)

print(f"Accuracy on test set:")
print(f"- Accuracy of KNN Classifier model on test dataset:              {acc_knn:.4f}")
print(f"- Accuracy of Decision Tree model on test dataset:    {acc_tree:.4f}")
print(f"- Accuracy of SVC model on test dataset:              {acc_svc:.4f}")
print(f"- Accuracy of Logistic Regression model on test dataset:    {acc_log:.4f}")
print(f"- Accuracy of Random Forest model on test dataset:    {acc_rf:.4f}")

Accuracy on test set:
- Accuracy of KNN Classifier model on test dataset:              0.6250
- Accuracy of Decision Tree model on test dataset:    0.6250
- Accuracy of SVC model on test dataset:              0.7500
- Accuracy of Logistic Regression model on test dataset:    0.7500
- Accuracy of Random Forest model on test dataset:    0.6250
